In [1]:
pip install editdistance

In [2]:
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
import json
import editdistance  # Standard library for computing CER and WER Levenshtein distance

# --- 1. DEVICE SETUP & LOAD VOCABULARY ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device for testing: {device}")

try:
    with open('ctc_vocab.json', 'r', encoding='utf-8') as f:
        idx_to_char = json.load(f)
    # Ensure dictionary keys are read as integers
    idx_to_char = {int(k): v for k, v in idx_to_char.items()}
    num_classes = len(idx_to_char)
except Exception as e:
    print(f"Error loading ctc_vocab.json: {e}")
    raise SystemExit("Ensure ctc_vocab.json matches the training output.")

# --- 2. REBUILD ARCHITECTURE ---
class HybridCTCModel(nn.Module):
    def __init__(self, num_classes):
        super(HybridCTCModel, self).__init__()
        resnet = timm.create_model('resnet18', pretrained=False)
        self.cnn = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.act1, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3
        )
        self.sequence_builder = nn.Conv2d(256, 256, kernel_size=(4, 1))
        self.transformer_layer = nn.TransformerEncoderLayer(d_model=256, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.transformer_layer, num_layers=2)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.sequence_builder(x)
        x = x.squeeze(2).permute(0, 2, 1)
        x = self.transformer(x)
        x = self.fc(x)
        return x.log_softmax(2).permute(1, 0, 2)

# Instantiate and Load your saved .pth file
model = HybridCTCModel(num_classes).to(device)
try:
    model.load_state_dict(torch.load('hybrid_ctc_model.pth', map_location=device))
    model.eval() # Freeze layers out of training mode
    print("Successfully loaded 'hybrid_ctc_model.pth' for testing.")
except Exception as e:
    print(f"Error loading weights: {e}")
    raise SystemExit("Make sure 'hybrid_ctc_model.pth' is present in this directory.")

# --- 3. DATASET & TEST LOADER SETUP ---
print("Loading testing split from Hugging Face...")
hf_dataset = load_dataset("c3rl/IIIT-INDIC-HW-WORDS-Hindi")
test_data = hf_dataset['test']

class HFWordDataset(Dataset):
    def __init__(self, hf_split, transform=None):
        self.hf_split = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.hf_split)

    def __getitem__(self, idx):
        item = self.hf_split[idx]
        image = item['image'].convert('RGB')
        text = item['text'] if item['text'] else ""
        if self.transform:
            image = self.transform(image)
        return image, text

transform = transforms.Compose([
    transforms.Resize((64, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = HFWordDataset(test_data, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# --- 4. CTC GREEDY DECODER FUNCTION ---
def decode_predictions(predictions):
    predicted_text = []
    previous_char_idx = -1
    for idx in predictions:
        idx = idx.item()
        if idx != 0 and idx != previous_char_idx:
            if idx in idx_to_char:
                predicted_text.append(idx_to_char[idx])
        previous_char_idx = idx
    return "".join(predicted_text)

# --- 5. EVALUATION LOOP & METRICS ENGINE ---
print("Running complete evaluation loop on Test Split...")

total_char_distance = 0
total_chars_in_ground_truth = 0
total_words_failed = 0
total_words_evaluated = 0

# Sample list to store a few visualization targets for review/PPT presentation
sample_visualizations = []

with torch.no_grad():
    for images, ground_truths in test_loader:
        images = images.to(device)
        preds = model(images) # Output shape: [24, Batch, Num_Classes]

        # Pull out the highest probability token for each slice
        _, max_indices = torch.max(preds, 2) # Shape: [24, Batch]
        max_indices = max_indices.permute(1, 0) # Swap axes back to [Batch, 24] for easy row looping

        for idx, row in enumerate(max_indices):
            predicted_word = decode_predictions(row)
            actual_word = ground_truths[idx]

            # Calculate Edit Distance at character level
            char_dist = editdistance.eval(predicted_word, actual_word)
            total_char_distance += char_dist
            total_chars_in_ground_truth += max(len(actual_word), 1) # Prevent division by zero

            # Calculate Word Match
            if predicted_word != actual_word:
                total_words_failed += 1
            total_words_evaluated += 1

            # Keep a few distinct examples for presentation visibility
            if len(sample_visualizations) < 5 and len(actual_word) > 2:
                sample_visualizations.append((actual_word, predicted_word))

# --- 6. CALCULATE FINAL RESULTS ---
final_cer = (total_char_distance / total_chars_in_ground_truth) * 100
final_wer = (total_words_failed / total_words_evaluated) * 100
word_accuracy = 100 - final_wer

print("\n=============================================")
print("             DEEPVISION AI METRICS           ")
print("=============================================")
print(f"Total Test Words Evaluated : {total_words_evaluated}")
print(f"Character Error Rate (CER) : {final_cer:.2f}%")
print(f"Word Error Rate (WER)      : {final_wer:.2f}%")
print(f"Overall Word Accuracy      : {word_accuracy:.2f}%")
print("=============================================\n")

print("📋 Sample Output Predictions (Ground Truth -> Predicted):")
for target, prediction in sample_visualizations:
    status = "✅ MATCH" if target == prediction else "❌ MISMATCH"
    print(f"  • Real String: '{target}'  |  Predicted: '{prediction}'  --> {status}")

Using device for testing: cuda
Successfully loaded 'hybrid_ctc_model.pth' for testing.
Loading testing split from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/461M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/461M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/459M [00:00<?, ?B/s]

... (2 outputs left out)

Generating train split:   0%|          | 0/69853 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/12708 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12869 [00:00<?, ? examples/s]

Running complete evaluation loop on Test Split...

             DEEPVISION AI METRICS           
Total Test Words Evaluated : 12869
Character Error Rate (CER) : 15.73%
Word Error Rate (WER)      : 53.27%
Overall Word Accuracy      : 46.73%

📋 Sample Output Predictions (Ground Truth -> Predicted):
  • Real String: 'अनाथों'  |  Predicted: 'अनाथों'  --> ✅ MATCH
  • Real String: 'बसर'  |  Predicted: 'बसर'  --> ✅ MATCH
  • Real String: 'मुझमें'  |  Predicted: 'मुझमें'  --> ✅ MATCH
  • Real String: 'एटीएमों'  |  Predicted: 'एटीएमों'  --> ✅ MATCH
  • Real String: 'अश्लील'  |  Predicted: 'अरलील'  --> ❌ MISMATCH


In [3]:
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
import json
import editdistance
import random # Added to randomize the output samples

# --- 1. DEVICE SETUP & LOAD VOCABULARY ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device for testing: {device}")

try:
    with open('ctc_vocab.json', 'r', encoding='utf-8') as f:
        idx_to_char = json.load(f)
    idx_to_char = {int(k): v for k, v in idx_to_char.items()}
    num_classes = len(idx_to_char)
except Exception as e:
    print(f"Error loading ctc_vocab.json: {e}")
    raise SystemExit("Ensure ctc_vocab.json matches the training output.")

# --- 2. REBUILD ARCHITECTURE ---
class HybridCTCModel(nn.Module):
    def __init__(self, num_classes):
        super(HybridCTCModel, self).__init__()
        resnet = timm.create_model('resnet18', pretrained=False)
        self.cnn = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.act1, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3
        )
        self.sequence_builder = nn.Conv2d(256, 256, kernel_size=(4, 1))
        self.transformer_layer = nn.TransformerEncoderLayer(d_model=256, nhead=8, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.transformer_layer, num_layers=2)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.sequence_builder(x)
        x = x.squeeze(2).permute(0, 2, 1)
        x = self.transformer(x)
        x = self.fc(x)
        return x.log_softmax(2).permute(1, 0, 2)

model = HybridCTCModel(num_classes).to(device)
try:
    model.load_state_dict(torch.load('hybrid_ctc_model.pth', map_location=device))
    model.eval()
    print("Successfully loaded 'hybrid_ctc_model.pth' for testing.")
except Exception as e:
    print(f"Error loading weights: {e}")
    raise SystemExit("Make sure 'hybrid_ctc_model.pth' is present in this directory.")

# --- 3. DATASET & TEST LOADER SETUP ---
print("Loading testing split from Hugging Face...")
hf_dataset = load_dataset("c3rl/IIIT-INDIC-HW-WORDS-Hindi")
test_data = hf_dataset['test']

class HFWordDataset(Dataset):
    def __init__(self, hf_split, transform=None):
        self.hf_split = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.hf_split)

    def __getitem__(self, idx):
        item = self.hf_split[idx]
        image = item['image'].convert('RGB')
        text = item['text'] if item['text'] else ""
        if self.transform:
            image = self.transform(image)
        return image, text

transform = transforms.Compose([
    transforms.Resize((64, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = HFWordDataset(test_data, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# --- 4. CTC GREEDY DECODER FUNCTION ---
def decode_predictions(predictions):
    predicted_text = []
    previous_char_idx = -1
    for idx in predictions:
        idx = idx.item()
        if idx != 0 and idx != previous_char_idx:
            if idx in idx_to_char:
                predicted_text.append(idx_to_char[idx])
        previous_char_idx = idx
    return "".join(predicted_text)

# --- 5. EVALUATION LOOP & METRICS ENGINE ---
print("Running complete evaluation loop on Test Split...")

total_char_distance = 0
total_chars_in_ground_truth = 0
total_words_failed = 0
total_words_evaluated = 0

# Lists to store ALL results so we can randomly sample them later
successful_matches = []
failed_matches = []

with torch.no_grad():
    for batch_idx, (images, ground_truths) in enumerate(test_loader):
        # Print progress so you know it's not frozen
        if (batch_idx + 1) % 100 == 0:
            print(f"Processed {(batch_idx + 1) * 32} images...")

        images = images.to(device)
        preds = model(images)

        _, max_indices = torch.max(preds, 2)
        max_indices = max_indices.permute(1, 0)

        for idx, row in enumerate(max_indices):
            predicted_word = decode_predictions(row)
            actual_word = ground_truths[idx]

            # Skip empty labels just in case
            if len(actual_word) == 0:
                continue

            char_dist = editdistance.eval(predicted_word, actual_word)
            total_char_distance += char_dist
            total_chars_in_ground_truth += len(actual_word)

            total_words_evaluated += 1

            # Categorize the prediction
            if predicted_word == actual_word:
                # Only keep words longer than 2 characters for better visual examples
                if len(actual_word) > 2:
                    successful_matches.append((actual_word, predicted_word))
            else:
                total_words_failed += 1
                if len(actual_word) > 2:
                    failed_matches.append((actual_word, predicted_word))

# --- 6. CALCULATE FINAL RESULTS ---
final_cer = (total_char_distance / total_chars_in_ground_truth) * 100
final_wer = (total_words_failed / total_words_evaluated) * 100
word_accuracy = 100 - final_wer

print("\n=============================================")
print("             DEEPVISION AI METRICS           ")
print("=============================================")
print(f"Total Test Words Evaluated : {total_words_evaluated}")
print(f"Character Error Rate (CER) : {final_cer:.2f}%")
print(f"Word Error Rate (WER)      : {final_wer:.2f}%")
print(f"Overall Word Accuracy      : {word_accuracy:.2f}%")
print("=============================================\n")

# --- 7. DISPLAY RANDOMIZED SAMPLES ---
print("✅ RANDOM SUCCESSFUL MATCHES (Ground Truth -> Predicted):")
# Grab 10 random matches
random_successes = random.sample(successful_matches, min(10, len(successful_matches)))
for target, prediction in random_successes:
    print(f"  • Real String: '{target}'  |  Predicted: '{prediction}'")

print("\n❌ RANDOM FAILED MATCHES (Ground Truth -> Predicted):")
# Grab 10 random mismatches
random_failures = random.sample(failed_matches, min(10, len(failed_matches)))
for target, prediction in random_failures:
    print(f"  • Real String: '{target}'  |  Predicted: '{prediction}'")

Using device for testing: cuda
Successfully loaded 'hybrid_ctc_model.pth' for testing.
Loading testing split from Hugging Face...
Running complete evaluation loop on Test Split...
Processed 3200 images...
Processed 6400 images...
Processed 9600 images...
Processed 12800 images...

             DEEPVISION AI METRICS           
Total Test Words Evaluated : 12869
Character Error Rate (CER) : 15.73%
Word Error Rate (WER)      : 53.27%
Overall Word Accuracy      : 46.73%

✅ RANDOM SUCCESSFUL MATCHES (Ground Truth -> Predicted):
  • Real String: 'शिकार'  |  Predicted: 'शिकार'
  • Real String: 'प्रवेश'  |  Predicted: 'प्रवेश'
  • Real String: 'परिष्कृत'  |  Predicted: 'परिष्कृत'
  • Real String: 'मुसहर।'  |  Predicted: 'मुसहर।'
  • Real String: 'एयरलाईंस'  |  Predicted: 'एयरलाईंस'
  • Real String: 'अश्वघोष'  |  Predicted: 'अश्वघोष'
  • Real String: 'पेपर'  |  Predicted: 'पेपर'
  • Real String: 'जिन्होने'  |  Predicted: 'जिन्होने'
  • Real String: 'मेकर्स'  |  Predicted: 'मेकर्स'
  • Real Stri